# rl_control

> Phase 12, Track B: the pure control logic of the RL loop (no GPU libraries): the learning rate steered to a target per-step policy change (with warm-up, clamping and undoing of runaway steps), the stopping rule, and micro-batching.

In [ ]:
#| default_exp rl_control

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np


def nanMean(x):
    x = [v for v in x if not np.isnan(v)]
    return float(np.mean(x)) if x else float('nan')


def warmLr(lr, s, warmup):
    "The learning rate of step s (from 0): rises linearly to lr over the first `warmup` steps."
    return lr * min(1.0, (s + 1) / warmup) if warmup else lr


def nextLr(lr, stepKL, cfg, warming=False, trained=True):
    """The KL-targeted learning rate (off when cfg.klTarget is 0): -> (next learning rate, undo this step). A step whose
    policy change exceeds rejectFactor x klTarget is undone and the rate halved; above 2 x klTarget the rate is halved,
    below klTarget / 2 raised by half (not while warming up, when the step used only part of the rate, and not after a
    step with nothing to train on); always within [lrMin, lrMax]."""
    if not cfg.klTarget or not trained:
        return lr, False
    reject = stepKL > cfg.rejectFactor * cfg.klTarget
    if reject or stepKL > 2 * cfg.klTarget:
        lr = lr / 2
    elif stepKL < cfg.klTarget / 2 and not warming:
        lr = lr * 1.5
    return float(min(cfg.lrMax, max(cfg.lrMin, lr))), reject


def stepLine(m, steps):
    "One line per step, to watch a run: learning rate, policy change, drift from the warm start, welfare, invalid replies."
    return (f"step {m['step']}/{steps}: lr {m['lr']:.2g}, step KL {m['stepKL']:.4f}, KL to warm start {m['kl']:.4f}, "
            f"welfare {m['welfare']:.3f}, invalid {m['invalid']:.3f}, undone {'yes' if m['rejected'] else 'no'}, "
            f"{m['rolloutSeconds'] + m['updateSeconds']:.0f} s")


def progressBar(total, start, desc, show=True):
    "A progress bar over a run's steps (tqdm: a widget in notebooks), starting at `start` on resume."
    from tqdm.auto import tqdm
    return tqdm(total=total, initial=start, desc=desc, unit='step', disable=not show)


def haltReason(history, cfg):
    "The stopping rule: invalid replies above haltInvalid for haltRun steps in a row, or the mean KL above haltKL."
    last = history[-cfg.haltRun:]
    if len(last) == cfg.haltRun and all(h['invalid'] > cfg.haltInvalid for h in last):
        return f"invalid replies above {cfg.haltInvalid:g} for {cfg.haltRun} steps (steps {last[0]['step']}-{last[-1]['step']})"
    if history and history[-1]['kl'] > cfg.haltKL:
        return f"mean KL {history[-1]['kl']:.3f} above {cfg.haltKL:g} at step {history[-1]['step']}"
    return None


def microBatches(samples, cfg):
    "Chunks of at most microBatch samples; with per-role adapters every chunk holds one slot's samples."
    key = (lambda x: x['slot']) if cfg.perRole else (lambda x: 0)
    groups = [[x for x in samples if key(x) == k] for k in sorted({key(x) for x in samples})]
    return [g[a:a + cfg.microBatch] for g in groups for a in range(0, len(g), cfg.microBatch)]

## Tests

In [ ]:
import math
from types import SimpleNamespace as C

m = dict(step=12, lr=2.2e-4, stepKL=0.0031, kl=0.041, welfare=0.2512, invalid=0.0, rejected=False, rolloutSeconds=60.2, updateSeconds=102.9)
assert stepLine(m, 60) == ('step 12/60: lr 0.00022, step KL 0.0031, KL to warm start 0.0410, welfare 0.251, invalid 0.000, '
                           'undone no, 163 s')
assert 'undone yes' in stepLine(dict(m, rejected=True), 60)
assert math.isnan(nanMean([float('nan')])) and nanMean([1.0, float('nan'), 3.0]) == 2.0
# warm-up: linear to lr over `warmup` steps, then lr
assert np.allclose([warmLr(1e-4, s, 4) for s in range(5)], [2.5e-5, 5e-5, 7.5e-5, 1e-4, 1e-4]) and warmLr(1e-4, 0, 0) == 1e-4
# the KL-targeted learning rate: steered to the per-step target, clamped, runaway steps undone; off without a target
c = C(klTarget=0.01, rejectFactor=4.0, lrMin=1e-6, lrMax=3e-4)
same = lambda a, b: math.isclose(a[0], b[0]) and a[1] == b[1]
assert same(nextLr(1e-4, 0.01, c), (1e-4, False))                    # in the band: unchanged
assert same(nextLr(1e-4, 0.025, c), (5e-5, False))                   # too far: halved
assert same(nextLr(1e-4, 0.002, c), (1.5e-4, False))                 # too little: raised by half
assert same(nextLr(1e-4, 0.05, c), (5e-5, True))                     # runaway: undone and halved
assert same(nextLr(2.5e-4, 0.001, c), (3e-4, False)) and same(nextLr(1.5e-6, 0.5, c), (1e-6, True))   # clamped
assert nextLr(1e-4, 9.9, C(klTarget=0.0)) == (1e-4, False)
# while warming up (the step used part of the rate) a small change does not raise it; too far still lowers or undoes
assert same(nextLr(1e-4, 0.002, c, warming=True), (1e-4, False)) and same(nextLr(1e-4, 0.05, c, warming=True), (5e-5, True))
# a step with nothing to train on (every group tied) leaves the rate alone
assert nextLr(1e-4, 0.0, c, trained=False) == (1e-4, False)
# the stopping rule: invalid replies too high for haltRun steps in a row, or the KL too high
h = lambda *xs: [dict(step=k + 1, invalid=v, kl=kl) for k, (v, kl) in enumerate(xs)]
c = C(haltRun=3, haltInvalid=0.10, haltKL=0.5)
assert haltReason(h((0.2, 0.0), (0.2, 0.0)), c) is None and haltReason(h((0.2, 0.0), (0.05, 0.0), (0.2, 0.0), (0.2, 0.0)), c) is None
assert 'steps 2-4' in haltReason(h((0.0, 0.0), (0.2, 0.0), (0.3, 0.0), (0.11, 0.0)), c)
assert 'KL 0.600' in haltReason(h((0.0, 0.1), (0.0, 0.6)), c) and haltReason([], c) is None
# micro-batches: per-role chunks never mix slots
mb = microBatches([dict(slot=s) for s in (0, 1, 0, 1, 1)], C(perRole=True, microBatch=2))
assert [[x['slot'] for x in b] for b in mb] == [[0, 0], [1, 1], [1]]
assert [len(b) for b in microBatches([dict(slot=s) for s in (0, 1, 0, 1, 1)], C(perRole=False, microBatch=2))] == [2, 2, 1]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()